# Query signal validation

This is an executed, exploratory check of the project direction, not a trained-model comparison. The test asks whether current queries share useful information with historical queries and candidate titles.

**Protocol:** seed 4222; uniform sample of 5,000 whole search records from all 173,831 records; preserve all sampled candidates. Scan the entire product catalog and retain metadata for sampled candidates, rather than sampling catalog rows. Missing metadata is reported separately.

The exact-repeat check uses **all records**. Other results use the stated sample. Term IDs are split on `\x18` (octal `\030`); lists of historical queries are split on `_`; `-1` is query-less history, not a query token. Set overlap ignores token order and repetitions; that is appropriate for these lexical diagnostics but may lose meaningful word order.

Source: [official JDSearch README](https://github.com/rucliujn/JDsearch#dataset-files). Matching shared IDs is possible; the original words cannot be recovered. This notebook does not choose an architecture.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
    # Clone/open the repository in Colab so the shared scripts module is available.
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/query_diagnostics.py").is_file()), None)
except ImportError:
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/query_diagnostics.py").is_file())
    DATA_DIR = REPO_ROOT / "data"
if REPO_ROOT is None:
    raise FileNotFoundError("Open the repository checkout containing scripts/query_diagnostics.py first.")
sys.path.insert(0, str(REPO_ROOT))
from scripts.query_diagnostics import run_validation, validate_helpers, expected_ndcg, bootstrap_mean
OUTPUT_DIR = REPO_ROOT / "outputs/interim/tables/query_validation"
SAMPLE_SIZE, SEED = 5000, 4222


## Validate parsing and evaluation

Assertions cover complete IDs versus digit substrings, short valid IDs, placeholders, reordered and repeated tokens, malformed encodings, empty positives, and score ties. An independent permutation calculation checks expected NDCG under ties.

The file stores positive candidates first, so preserving original order when scores tie would leak labels. Every score below uses **expected performance under random tie breaking**. Relevance gains are linear 0/1/2/3; all-zero lists are excluded from ranking means. These exploratory choices are explicit and need not be the final evaluation contract.


In [2]:
from itertools import permutations
validate_helpers()
labels = np.array([3, 0, 1, 2])
scores = np.array([1, 1, 0, 0])
discount = 1 / np.log2(np.arange(2, 5))
ideal = np.dot(np.sort(labels)[::-1][:3], discount)
orders = [p for p in permutations(range(4)) if all(scores[p[j]] >= scores[p[j + 1]] for j in range(3))]
independent = np.mean([np.dot(labels[list(p)][:3], discount) / ideal for p in orders])
assert np.isclose(independent, expected_ndcg(scores, labels, 3))
print("Parser assertions and independent permutation audit passed.")


Parser assertions and independent permutation audit passed.


## Run the diagnostics

**Title query coverage** = shared distinct terms / distinct terms in the current query. **Jaccard** = shared distinct terms / union of distinct terms. A query shuffled from another sampled record is a control for coincidental overlap; it is not a recommendation baseline.

Historical-query overlap describes supplied actions, not distinct search episodes: several products may be interacted with under one issued query. The history scores count past interactions with the exact candidate product, using either all supplied history or only actions whose recorded query overlaps the current query. The latter is a deliberately simple hard filter, not an attention model.


In [3]:
summary, by_label, legacy_disagreements, ranking, per_search = run_validation(
    DATA_DIR, OUTPUT_DIR, sample_size=SAMPLE_SIZE, seed=SEED
)


Sample: 5,000/173,831 records; 414,359 distinct candidate IDs
Catalog scan: 12,141,247 rows; found 412,433/414,359 sampled items
{
  "seed": 4222,
  "total_records": 173831,
  "sample_records": 5000,
  "sample_candidate_occurrences": 449052,
  "metadata_catalog_rows_scanned": 12141247,
  "candidate_metadata_coverage": 0.9956352493697834,
  "all_zero_sample_records": 59,
  "share_sample_records_with_search_history": 0.9694,
  "raw_exact_repeat_records": 0,
  "set_exact_repeat_records": 15,
  "full_data_raw_exact_repeat_records": 1,
  "full_data_distinct_token_set_repeat_records": 610,
  "full_data_token_set_repeat_share": 0.003509155444080745,
  "repeat_examples_order_or_multiplicity_differences": [
    {
      "current": [
        "70033454",
        "18321627"
      ],
      "past": [
        "18321627",
        "70033454"
      ]
    },
    {
      "current": [
        "13491834",
        "50856089"
      ],
      "past": [
        "50856089",
        "13491834"
      ]
    },
    {


In [4]:
display(by_label)
display(legacy_disagreements)
display(ranking)
print("Share of sampled lists with varying title coverage:", f"{per_search.title_coverage_varies.mean():.2%}")
print("Per-search paired history-filter difference, 95% bootstrap interval:",
      bootstrap_mean(per_search.ndcg_matching_query_history_item_count - per_search.ndcg_history_item_count))


Share of sampled lists with varying title coverage: 90.00%
Per-search paired history-filter difference, 95% bootstrap interval: (-0.049333265914409945, -0.055171799904628324, -0.04372870471348071)


,candidate_count,title_shared_terms,title_query_coverage,title_jaccard,brand_any_token,category3_any_token,shuffled_query_title_coverage,history_item_count,matching_query_history_item_count
label,,,,,,,,,
0,437794,1.799237,0.720880,0.076671,0.183634,0.324255,0.025013,0.013952,0.003764
1,6564,1.938757,0.723270,0.085618,0.196222,0.366088,0.025794,0.203534,0.061700
2,2135,1.916159,0.730583,0.083311,0.213115,0.341920,0.022690,0.458548,0.126464
3,599,1.731219,0.739461,0.085387,0.158598,0.355593,0.018516,0.467446,0.086811


,legacy_only,token_only
brand,0,64099
category3,0,70937


,evaluated_queries,mean_ndcg_at_10,delta_vs_random,delta_ci_low,delta_ci_high
score,,,,,
title_coverage,4941,0.091701,0.010828,0.008168,0.013515
title_jaccard,4941,0.094762,0.013889,0.009047,0.018618
history_item_count,4941,0.180903,0.100030,0.092532,0.107676
matching_query_history_item_count,4941,0.131570,0.050697,0.045144,0.056183


## Interpretation and limits

Use the saved values to assess signal availability, not to promise revenue gains or superiority over last year's project. NDCG comparisons here are untrained heuristics on an exploratory sample, with paired bootstrap confidence intervals over search records (2,000 resamples).

A small title-overlap gain indicates a lexical relevance feature is usable, while leaving room for history and other metadata. A negative result for the hard history filter would argue against throwing away query-less or lexically unrelated history automatically; it would not rule out learning softer weights. Exact token overlap misses related queries with no shared terms.

The main interim notebook has corrected brand/category matching, but its affected statistics must be rerun before quoting them. This notebook's sample results must not be presented as full-data statistics, except the explicitly named full-data repeat check.


## Findings from this run

- Full-data repeat audit: 1 raw exact match; 610 records (0.35%) with the same distinct term set. Token-order differences explain inspected examples.
- 41.14% of the 5,000 sampled records have some current/history query-term overlap. Exact repetition is too rare to anchor the project.
- Title coverage varies within 90.00% of sampled lists among candidates with metadata. The lexical signal is usable for ranking.
- Query-title Jaccard ranking: NDCG@10 = 0.0948 versus 0.0809 for expected random order; paired difference interval = [0.0090, 0.0186].
- Exact-product history count ranks at 0.1809; limiting that count to overlapping-query history ranks at 0.1316. The paired difference interval is [-0.0552, -0.0437]. A hard history filter is not supported by this check.

These are exploratory, untrained scores on 4,941 positive-containing sampled searches. They do not establish superiority over a learned model or last year's project. Keep architecture open and investigate softer weighting, query-less history and metadata matches. See `docs/FINDINGS.md` for the protocol, checks and limitations.
